# Hito 3 — Metodología y primeros resultados
## Colombia / Educación Superior / Gobierno Nacional 2022–2026

**Equipo:** Juan Jacobo Delgado · Gabriel Martinez · Nicolas Cuaran · Juan Jose Orozco · Sebastian Belalcazar  
**Criterios rúbrica:** D (Metodología, 20 pts) + F (Validación e incertidumbre, 10 pts)

---
### Estructura del análisis
1. Carga de datos desde PostgreSQL (star schema SNIES 2018–2024)
2. Análisis descriptivo de tendencias pre/post 2022
3. Series de Tiempo Interrumpidas (ITS) — modelo segmentado
4. Diferencias en Diferencias (DiD) — sector Oficial vs. Privada
5. Bootstrap e intervalos de confianza
6. Análisis de sensibilidad y placebos
7. Resumen de hallazgos (neutralidad técnica)

In [ ]:
# Configuración de rutas y dependencias
import sys
from pathlib import Path

# Asegurar que el root del proyecto esté en el path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm

pio.renderers.default = 'notebook'
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)

print('✓ Dependencias cargadas')

---
## 1. Carga de datos

Los datos provienen del star schema PostgreSQL construido en el Hito 2:
- **`facts.fact_estudiantes`**: matriculados, inscritos, admitidos, primer_curso, graduados
- **`facts.dim_institucion`**: sector IES (Oficial / Privada), carácter
- **`facts.dim_tiempo`**: año y semestre
- **`facts.dim_geografia`**: departamento y municipio

In [ ]:
from analysis.queries import (
    get_matricula_por_sector,
    get_panel_ies,
    get_embudo_estudiantil,
    get_matricula_por_departamento,
)

# Cargar series temporales por sector
df_mat = get_matricula_por_sector('matriculados')
df_pc  = get_matricula_por_sector('primer_curso')
df_gr  = get_matricula_por_sector('graduados')
df_panel = get_panel_ies()
df_embudo = get_embudo_estudiantil()

print(f'Matriculados:  {len(df_mat)} filas, periodos: {df_mat["periodo"].nunique()}')
print(f'Primer curso:  {len(df_pc)} filas')
print(f'Graduados:     {len(df_gr)} filas')
print(f'Panel IES:     {df_panel["codigo_ies"].nunique()} IES, {len(df_panel)} obs')
print(f'\nPeriodos disponibles:')
print(sorted(df_mat['periodo'].unique()))

---
## 2. Análisis de tendencias pre/post 2022

In [ ]:
from analysis.tendencias import run_tendencias

# Matriculados
res_tend = run_tendencias(df_mat, tipo_evento='matriculados')
res_tend['fig_serie'].show()

In [ ]:
# Resumen cuantitativo pre/post
print('=== Cambio pre/post 2022 por sector ===')
print(res_tend['resumen'].to_string(index=False))

In [ ]:
# Gráfico variación anual
res_tend['fig_variacion'].show()

In [ ]:
# Embudo estudiantil: tasas de conversión
df_emb_of = df_embudo[df_embudo['sector_ies'] == 'Oficial'].copy()
df_piv = df_emb_of.pivot_table(index='periodo', columns='tipo_evento', values='total', aggfunc='sum')
df_piv = df_piv.sort_index()

# Tasa admisión/inscripción
if 'admitidos' in df_piv.columns and 'inscritos' in df_piv.columns:
    df_piv['tasa_admision'] = df_piv['admitidos'] / df_piv['inscritos'] * 100
# Tasa matriculación
if 'matriculados' in df_piv.columns and 'admitidos' in df_piv.columns:
    df_piv['tasa_matriculacion'] = df_piv['matriculados'] / df_piv['admitidos'] * 100

print('=== Embudo estudiantil — IES Oficiales ===')
print(df_piv.round(1).to_string())

---
## 3. Series de Tiempo Interrumpidas (ITS)

**Modelo:**
$$Y_t = \alpha_0 + \alpha_1 t + \alpha_2 D_t + \alpha_3 (t - T_0) D_t + \varepsilon_t$$

donde $D_t = 1$ si $t \geq T_0 = 2022\text{-S2}$.

- $\alpha_2$: cambio **inmediato de nivel** al inicio del gobierno
- $\alpha_3$: cambio en la **pendiente** (tasa de crecimiento) post-2022

Errores HAC (Newey-West, maxlags=2) para corregir autocorrelación.

In [ ]:
from analysis.its import run_its

# ITS — Sector Oficial, matriculados
# Placebos: 2 y 4 semestres antes del punto de quiebre real
res_its_of = run_its(
    df_mat,
    sector='Oficial',
    tipo_evento='matriculados',
    placebos_t0=[6, 8],  # t=6 ≈ 2020-S2, t=8 ≈ 2021-S2
)

res_its_of['fig'].show()

In [ ]:
# Tabla de coeficientes ITS
coefs = res_its_of['coeficientes']
filas = []
for nombre, vals in coefs.items():
    filas.append({
        'Parámetro': nombre,
        'Estimado': vals['estimado'],
        'IC 95% inferior': vals['ic_95_lower'],
        'IC 95% superior': vals['ic_95_upper'],
        'p-value': vals['p_value'],
        'Significativo': '✓' if vals['p_value'] < 0.05 else '✗',
    })
df_coefs = pd.DataFrame(filas)
print('=== Coeficientes ITS — Oficial, matriculados ===')
print(df_coefs.to_string(index=False))

In [ ]:
# Bondad de ajuste y Chow
bondad = res_its_of['bondad_ajuste']
chow = res_its_of['chow']
print(f"R²={bondad['R2']} | R²ajust={bondad['R2_ajustado']} | AIC={bondad['AIC']} | DW={bondad['durbin_watson']}")
print(f"Chow: F={chow['chow_F']}, p={chow['chow_p']} → {chow['conclusion']}")

In [ ]:
# ITS — Sector Privada (grupo de comparación)
res_its_pr = run_its(df_mat, sector='Privada', tipo_evento='matriculados')
res_its_pr['fig'].show()

In [ ]:
# ITS — Primera matrícula (proxy de nuevos accesos)
res_its_pc = run_its(df_pc, sector='Oficial', tipo_evento='primer_curso')
res_its_pc['fig'].show()

---
## 4. Diferencias en Diferencias (DiD)

**Modelo agregado (sector × semestre):**
$$Y_{st} = \alpha + \beta_1 \text{POST}_t + \beta_2 \text{OFICIAL}_s + \beta_3 (\text{POST}_t \times \text{OFICIAL}_s) + \varepsilon_{st}$$

**Tratado:** IES Oficiales  
**Control:** IES Privadas  
**Pre:** 2018-S1 → 2022-S1  
**Post:** 2022-S2 → 2024-S2  

In [ ]:
from analysis.did import run_did_agregado, run_did_panel, run_event_study

# DiD Agregado
res_did = run_did_agregado(df_mat, tipo_evento='matriculados')
res_did['fig'].show()

In [ ]:
# Resultados DiD
r = res_did['resultados']
est = r['estimador_did']
med = r['medias']

print('=== Estimador DiD — matriculados ===')
print(f"β₃ = {est['beta_3']:,.0f}  IC95%=[{est['ic_95_lower']:,.0f}, {est['ic_95_upper']:,.0f}]  p={est['p_value']}")
print(f"Significativo: {est['significativo']}")
print()
print('=== Tabla 2x2 (medias por semestre) ===')
tabla = pd.DataFrame({
    'Sector': ['Oficial', 'Privada', 'Δ (Oficial − Privada)'],
    'Pre-2022': [med['oficial_pre'], med['privada_pre'], med['oficial_pre'] - med['privada_pre']],
    'Post-2022': [med['oficial_post'], med['privada_post'], med['oficial_post'] - med['privada_post']],
    'Δ (Post − Pre)': [
        med['oficial_post'] - med['oficial_pre'],
        med['privada_post'] - med['privada_pre'],
        med['did_manual'],
    ],
})
print(tabla.to_string(index=False))

In [ ]:
# DiD Panel con efectos fijos de IES
res_did_panel = run_did_panel(df_panel, tipo_evento='matriculados')
rp = res_did_panel.get('resultados', {})
ep = rp.get('estimador_did', {})

print('=== DiD Panel TWFE — matriculados ===')
print(f"β = {ep.get('beta'):.4f}  ≈ {ep.get('efecto_pct_aprox')}%")
print(f"IC95% = [{ep.get('ic_95_lower_beta'):.4f}, {ep.get('ic_95_upper_beta'):.4f}]")
print(f"p = {ep.get('p_value')}")
print(f"N IES = {rp.get('n_ies')}, N obs = {rp.get('n_obs')}")
print(f"\n{ep.get('interpretacion')}")

In [ ]:
# Event Study — pre-tendencias
res_es = run_event_study(df_mat, tipo_evento='matriculados')
res_es['fig'].show()

print('\n=== Test de pre-tendencias ===')
print(res_es['resultado']['conclusion_pretendencias'])
print(f"p-value test conjunto = {res_es['resultado']['test_pretendencias_p']}")

---
## 5. Bootstrap e intervalos de confianza

**Método:** Block bootstrap (bloque = 2 semestres), N = 1,000 re-muestras  
**IC:** Método de percentiles (2.5% y 97.5%)  
**Objetivo:** Cuantificar la incertidumbre de los estimadores ITS y DiD sin asumir distribución paramétrica

In [ ]:
from analysis.bootstrap import run_bootstrap_completo

# Identificar t0
t0_mask = (df_mat['ano'] == 2022) & (df_mat['semestre'] == 2)
t0 = int(df_mat.loc[t0_mask, 't'].drop_duplicates().iloc[0])
print(f'T₀ = {t0} (2022-S2)')

res_boot = run_bootstrap_completo(df_mat, t0=t0, tipo_evento='matriculados', n_boot=1000)

In [ ]:
# Distribución bootstrap α₂
res_boot['fig_alpha2'].show()

In [ ]:
# Distribución bootstrap β₃
res_boot['fig_beta3'].show()

In [ ]:
# Análisis de escenarios
boot_res = res_boot['resultado']
esc = boot_res['escenarios_oficial']
df_esc = pd.DataFrame(esc['escenarios'])

fig_esc = go.Figure()
for col, name, color, dash in [
    ('observado', 'Observado', '#1f77b4', 'solid'),
    ('escenario_base', 'Base (contrafactual)', 'gray', 'dash'),
    ('escenario_optimista', 'Optimista (+1σ)', 'green', 'dot'),
    ('escenario_adverso', 'Adverso (−1σ)', 'red', 'dot'),
]:
    fig_esc.add_trace(go.Scatter(x=df_esc['periodo'], y=df_esc[col],
                                 name=name, mode='lines+markers',
                                 line=dict(color=color, dash=dash)))

fig_esc.update_layout(
    title='Análisis de escenarios — IES Oficiales (matriculados)',
    xaxis_title='Semestre', yaxis_title='Estudiantes',
    template='plotly_white', height=430, hovermode='x unified',
)
fig_esc.show()

---
## 6. Análisis de sensibilidad

Variamos los supuestos clave para evaluar la robustez de los estimadores.

In [ ]:
# Sensibilidad al punto de quiebre
from analysis.its import run_its
import warnings

resultados_sensibilidad = []
for t_prueba, label in [(8, '2021-S2'), (9, '2022-S1'), (10, '2022-S2 (base)'), (11, '2023-S1')]:
    df_s = df_mat[df_mat['sector_ies'] == 'Oficial'].sort_values('t').copy()
    df_s['D'] = (df_s['t'] >= t_prueba).astype(int)
    df_s['t_post'] = (df_s['t'] - t_prueba) * df_s['D']
    X = sm.add_constant(df_s[['t', 'D', 't_post']])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mod = sm.OLS(df_s['total'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 2})
    ci = mod.conf_int()
    resultados_sensibilidad.append({
        'T₀ alternativo': label,
        'α₂ (cambio nivel)': round(mod.params['D'], 0),
        'IC inf': round(ci.loc['D', 0], 0),
        'IC sup': round(ci.loc['D', 1], 0),
        'p α₂': round(mod.pvalues['D'], 4),
        'α₃ (cambio tend)': round(mod.params['t_post'], 0),
        'p α₃': round(mod.pvalues['t_post'], 4),
    })

df_sens = pd.DataFrame(resultados_sensibilidad)
print('=== Sensibilidad al punto de quiebre — ITS, Oficial, matriculados ===')
print(df_sens.to_string(index=False))

In [ ]:
# Sensibilidad a la muestra de IES (DiD Panel)
from analysis.did import run_did_panel

for min_est, label in [(0, 'Todas las IES'), (500, 'IES ≥500 estudiantes'), (1000, 'IES ≥1000')]:
    df_filt = df_panel[df_panel.groupby('codigo_ies')['matriculados'].transform('max') >= min_est].copy()
    res = run_did_panel(df_filt, tipo_evento='matriculados')
    ep = res.get('resultados', {}).get('estimador_did', {})
    print(f"[{label}] β={ep.get('beta', 'N/A'):.4f} ({ep.get('efecto_pct_aprox', 'N/A')}%) "
          f"p={ep.get('p_value', 'N/A')}  N_IES={res.get('resultados', {}).get('n_ies', '?')}")

---
## 7. Resumen de hallazgos

### Interpretación técnicamente neutral (Criterio G)

| Tipo de afirmación | Contenido |
|---|---|
| **Hecho medido** | La matrícula en IES Oficiales registró cambios observables en la serie 2018–2024 |
| **Inferencia condicionada** | El modelo ITS detecta un cambio de nivel α₂ en 2022-S2; el DiD estima un diferencial β₃ respecto a las IES privadas |
| **Límite explícito** | No se puede descartar que la recuperación post-COVID explique parte del cambio observado |

> **Nota:** Los estimadores son medidas de *asociación*, no de *causalidad pura*. 
> La atribución directa requeriría un diseño experimental o una discontinuidad territorial no disponible con estos datos.

In [ ]:
# Ejecutar runner completo (guarda todos los resultados en data/results/)
# Descomentar para ejecución completa:
# from analysis.runner import run
# resultados = run()

# Cargar resumen ejecutivo si ya fue generado
import json
from config.globals import DATA_DIR

resumen_path = DATA_DIR / 'results' / 'resumen_ejecutivo_hito3.json'
if resumen_path.exists():
    with open(resumen_path, encoding='utf-8') as f:
        resumen = json.load(f)
    print(f"Resumen generado el: {resumen['fecha_ejecucion']}")
    print(f"Hallazgos registrados: {len(resumen['hallazgos_principales'])}")
    for h in resumen['hallazgos_principales']:
        sig = '✓' if h.get('significativo') else ('✗' if h.get('significativo') is False else '-')
        print(f"  [{sig}] {h['analisis']}: estimado={h.get('estimado')}")
else:
    print('Resumen no encontrado. Ejecuta: from analysis.runner import run; run()')

---
## Referencias metodológicas

- Bernal, R. & Peña, X. (2011). *Guía práctica para la evaluación de impacto*. Cap. 4 (DiD).
- Shadish, W., Cook, T. & Campbell, D. (2002). *Experimental and Quasi-Experimental Designs*. Cap. 7 (ITS).
- DNP (2022). *Plan Nacional de Desarrollo 2022–2026 — Colombia Potencia Mundial de la Vida*.
- MEN / SNIES: bases de datos de matrícula, 2018–2024.
- Cameron, A. & Trivedi, P. (2005). *Microeconometrics*, Cap. 11 (Bootstrap).

---
*Documento generado para el Hito 3 del Seminario de Ingeniería de Datos e IA — UAO*